In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn import tree

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
X_train=train_df.drop(["index","name","health_status"],axis=1)
X_test=test_df.drop(["index","name"],axis=1)

In [ ]:
# 'audio_l'과 'audio_r'에서 'N'을 1로, 'A'를 0으로 변경
X_train['audio_l'] = X_train['audio_l'].replace({'N': 0, 'A': 1}).astype(int)
X_train['audio_r'] = X_train['audio_r'].replace({'N': 0, 'A': 1}).astype(int)

X_test['audio_l'] = X_test['audio_l'].replace({'N': 0, 'A': 1}).astype(int)
X_test['audio_r'] = X_test['audio_r'].replace({'N': 0, 'A': 1}).astype(int)


# 'audio_l'과 'audio_r'의 OR 연산 결과로 'audio' 컬럼 생성
X_train['audio'] = X_train['audio_l'] | X_train['audio_r']
X_test['audio'] = X_test['audio_l'] | X_test['audio_r']


X_train['bmi']= X_train.mass_kg/((X_train.stature_inch+100)/100)**2
X_train['age']= 2024-X_train.birth+1
X_train['bp']= (X_train.bp_high-X_train.bp_low)
X_train['bp_tf']=(X_train.bp_high>130)&(X_train.bp_low>85)
X_train['audio'] = X_train['audio_l'] | X_train['audio_r']
X_train["Waist_to_Height_Ratio"] = X_train["midsection_inch"] / X_train["stature_inch"] #허리-키 비율(WtHR)
X_train["LDL_to_HDL_Ratio"] = X_train["chol_bad"] / X_train["chol_good"] #LDL/HDL 비율
X_train["Total_Cholesterol"] = X_train["chol_good"] + X_train["chol_bad"]
X_train["Enzyme_Avg"] = X_train[["enzyme_1", "enzyme_2", "enzyme_3"]].mean(axis=1) #면역 효소 지표 평균
X_train["Vision_Imbalance"] = (X_train["vision_l"] != X_train["vision_r"])
X_train["Metabolic_Syndrome"] = (
    (X_train["bp_high"] > 130) |
    (X_train["bp_low"] > 85) |
    (X_train["chol_bad"] > 100) |
    (X_train["midsection_inch"] > 35) |
    (X_train["sugar_lvl"] > 100)
)

X_test['bmi']= X_test.mass_kg/((X_test.stature_inch+100)/100)**2
X_test['age']= 2024-X_test.birth+1
X_test['bp']= (X_test.bp_high-X_test.bp_low)
X_test['bp_tf']=(X_test.bp_high>130)&(X_test.bp_low>85)
X_test['audio'] =X_test['audio_l'] | X_test['audio_r']
X_test["Waist_to_Height_Ratio"] = X_test["midsection_inch"] / X_test["stature_inch"]
X_test["LDL_to_HDL_Ratio"] = X_test["chol_bad"] / X_test["chol_good"]
X_test["Total_Cholesterol"] = X_test["chol_good"] + X_test["chol_bad"]
X_test["Enzyme_Avg"] = X_test[["enzyme_1", "enzyme_2", "enzyme_3"]].mean(axis=1) #면역 효소 지표 평균
X_test["Vision_Imbalance"] = (X_test["vision_l"] != X_test["vision_r"])
X_test["Metabolic_Syndrome"] = (
    (X_test["bp_high"] > 130) |
    (X_test["bp_low"] > 85) |
    (X_test["chol_bad"] > 100) |
    (X_test["midsection_inch"] > 35) |
    (X_test["sugar_lvl"] > 100)
)

# drop_colums=['audio_l','audio_r']
# X_train=X_train.drop(drop_colums,axis=1)
# X_test=X_test.drop(drop_colums,axis=1)

# y_train=train_df['health_status']

In [ ]:
y_train=train_df['health_status']

In [ ]:
drop_features = [
    'immune_index', 'audio', 'bp_tf', 'enzyme_1', 
    'chol_bad', 'vision_l', 'urine_marker', 
    'sugar_lvl', 'lipid_lvl'
]

# 피처 제거
X_train= X_train.drop(columns=drop_features)
X_test=X_test.drop(columns=drop_features)

In [ ]:
# 주의: one-hot encoding을 하는 경우 train과 test에 동일하게 적용되어야 함
X = pd.concat([X_train, X_test], axis=0)
X = pd.get_dummies(X)
X_train = X.iloc[:len(X_train)]
X_test = X.iloc[len(X_train):]

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import KFold
# from sklearn.preprocessing import StandardScaler

# # 상관계수의 절댓값을 계산
# correlations = X_train.corrwith(y_train).abs()

# # 가중치로 사용하기 위해 상관계수의 절댓값을 정의
# weights = correlations.values


In [ ]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# scaler.fit(X_train)
# scaled_X_train=scaler.transform(X_train)
# scaled_X_test=scaler.transform(X_test)
    
# # 상관계수 가중치를 적용하여 스케일링된 데이터를 가중치와 곱해줌
# X_train_weighted = scaled_X_train

# # 다시 DataFrame으로 변환 (기존 열 이름 유지)
# X_train_weighted = pd.DataFrame(X_train_weighted, columns=X_train.columns)


In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score, roc_auc_score
# from sklearn.model_selection import KFold
# from sklearn.preprocessing import StandardScaler
# import pandas as pd

# # 데이터 스케일링
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)

# # 스케일링된 데이터를 다시 DataFrame으로 변환 (기존 열 이름 유지)
# X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

# kf = KFold(n_splits=5)
# splits = kf.split(X_train_scaled, y_train)

# f1_scores = []
# auc_roc_scores = []

# for i, (train_index, test_index) in enumerate(splits):
#     # 각 폴드의 훈련/검증 데이터 분할 (iloc 사용)
#     X_train_fold, y_train_fold = X_train_weighted.iloc[train_index], y_train.iloc[train_index]
#     X_test_fold, y_test_fold =X_train_weighted.iloc[test_index], y_train.iloc[test_index]

#     # 로지스틱 회귀 모델 학습
#     model = LogisticRegression(
#         penalty='l2',          # 'l1' 또는 'l2' 규제를 사용할 수 있습니다. 기본값은 'l2'입니다.
#         class_weight='balanced',
#         random_state=100,  
#         solver='lbfgs',    # 최적화
#         max_iter=1000          # 솔버가 수렴할 때까지 반복할 수 있는 최대 횟수
#     )
#     model.fit(X_train_fold, y_train_fold)

#     # 검증 데이터로 예측 수행
#     y_pred_fold = model.predict(X_test_fold)
#     y_pred_prob_fold = model.predict_proba(X_test_fold)[:, 1]

#     # 성능 평가 (F1 점수 및 AUC-ROC 점수)
#     f1 = f1_score(y_test_fold, y_pred_fold)
#     roc_auc = roc_auc_score(y_test_fold, y_pred_prob_fold)

#     f1_scores.append(f1)
#     auc_roc_scores.append(roc_auc)

#     # 각 폴드의 성능 출력
#     print(f"Fold {i + 1}: F1 Score = {f1:.4f}, AUC-ROC Score = {roc_auc:.4f}")

# # 전체 폴드에 대한 평균 성능 출력
# print(f"\nAverage F1 Score: {sum(f1_scores) / len(f1_scores):.4f}")
# print(f"Average AUC-ROC Score: {sum(auc_roc_scores) / len(auc_roc_scores):.4f}")


In [ ]:
model = LogisticRegression(
        penalty='l2',          # 'l1' 또는 'l2' 규제를 사용할 수 있습니다. 기본값은 'l2'입니다.
        class_weight='balanced',     # 클래스별 가중치를 설정합니다. None 또는 'balanced'를 사용할 수 있습니다.
        random_state=10,     # 무작위성을 제어하기 위한 시드 값입니다.
        solver='lbfgs',    # 최적화에 사용할 알고리즘입니다.
        max_iter=20000          # 솔버가 수렴할 때까지 반복할 수 있는 최대 횟수입니다.
    )
model.fit(X_train, y_train)

##### 매개변수	설명	기본값
1. penalty  :   규제(regularization)의 종류를 지정합니다. 'l1'(라쏘), 'l2'(릿지), 'elasticnet', 'none'을 사용할 수 있습니다.	'l2'
2. dual :   이중 문제(dual problem)를 풀지 여부를 지정합니다. 작은 데이터셋에서 'liblinear' 솔버와 함께 사용할 수 있습니다.	False
3. tol	:   알고리즘의 수렴 조건을 설정하는 허용 오차입니다.	1e-4
4. C	:   규제 강도를 제어하는 매개변수입니다. 작은 값일수록 강한 규제를 의미합니다. 규제 강도는 1 / C로 적용됩니다.	1.0
5. fit_intercept    :	상수 항을 모델에 포함할지 여부를 설정합니다.	True
6. intercept_scaling    :	상수 항의 스케일링을 설정합니다. 'liblinear' 솔버와 함께 사용할 때만 유효합니다.	1
7. class_weight     :	클래스별 가중치를 지정합니다. 'balanced' 또는 사전(dict) 형태로 설정할 수 있습니다.	None
8. random_state     :	무작위성을 제어하기 위한 시드 값입니다. 재현 가능한 결과를 위해 설정합니다.	None
9. solver   :   최적화를 수행할 알고리즘을 지정합니다. 'newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga' 중 선택 가능합니다.	'lbfgs'
10. max_iter    :   솔버가 수렴하기 위해 반복할 수 있는 최대 횟수를 설정합니다.	100
11. multi_class     :   다중 클래스 문제를 어떻게 설정할지 지정합니다. 'ovr'(One-vs-Rest), 'multinomial'(다중 분류), 'auto'(자동 선택)를 사용할 수 있습니다.	'auto'
12. verbose	    :   솔버 메시지의 출력 상세 수준을 설정합니다. 값이 클수록 더 많은 정보를 출력합니다. 일부 솔버에서만 사용 가능합니다.	0
13. warm_start     :    이전 학습 결과를 초기화하지 않고 이어서 학습할지 여부를 설정합니다. 점진적인 모델 업데이트에 유용합니다.	False
14. n_jobs    :   사용할 CPU 코어 수를 지정합니다. -1로 설정하면 모든 코어를 사용합니다. 일부 솔버에서만 사용할 수 있습니다.	None
15. l1_ratio    :    penalty='elasticnet'일 때만 사용됩니다. L1과 L2 규제의 혼합 비율을 설정하며, 0은 L2, 1은 L1을 의미합니다.	None


In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score, roc_auc_score
# from sklearn.model_selection import KFold

# kf = KFold(n_splits=5)
# splits = kf.split(X_train, y_train)

# f1_scores = []
# auc_roc_scores = []

# for i, (train_index, test_index) in enumerate(splits):
#     # 각 폴드의 훈련/검증 데이터 분할 (iloc 사용)
#     X_train_fold, y_train_fold = X_train.iloc[train_index], y_train.iloc[train_index]
#     X_test_fold, y_test_fold = X_train.iloc[test_index], y_train.iloc[test_index]

#     # 로지스틱 회귀 모델 학습
#     model = LogisticRegression(
#     penalty='l1',          # 'l1' 또는 'l2' 규제를 사용할 수 있습니다. 기본값은 'l2'입니다.
#     dual=False,            # liblinear 솔버에서는 dual=False만 지원합니다.
#     tol=1e-4,              # 수렴 조건을 설정하는 허용 오차입니다.
#     C=1.0,                 # 규제 강도를 제어하는 매개변수입니다. 값이 작을수록 규제가 강해집니다.
#     fit_intercept=True,    # 상수 항을 모델에 포함할지 여부입니다.
#     intercept_scaling=1,   # 상수 항 스케일링을 설정합니다. liblinear에서만 사용됩니다.
#     class_weight=None,     # 클래스별 가중치를 설정합니다. None 또는 'balanced'를 사용할 수 있습니다.
#     random_state=None,     # 무작위성을 제어하기 위한 시드 값입니다.
#     solver='liblinear',    # 최적화에 사용할 알고리즘입니다.
#     max_iter=100,          # 솔버가 수렴할 때까지 반복할 수 있는 최대 횟수입니다.
#     multi_class='ovr',     # 다중 클래스 문제에서 One-vs-Rest 방식을 사용합니다.
#     verbose=0,             # 솔버의 출력 메시지 상세 수준입니다. 0으로 설정하면 출력이 없습니다.
#     warm_start=False,      # 이전 학습 결과를 초기화하지 않고 이어서 학습할지 여부입니다.
#     n_jobs=None            # 병렬 처리를 위한 CPU 코어 수 설정입니다. liblinear는 병렬 처리를 지원하지 않으므로 None입니다.
# )
#     model.fit(X_train_fold, y_train_fold)

#     # 검증 데이터로 예측 수행
#     y_pred_fold = model.predict(X_test_fold)
#     y_pred_prob_fold = model.predict_proba(X_test_fold)[:, 1]

#     # 성능 평가 (F1 점수 및 AUC-ROC 점수)
#     f1 = f1_score(y_test_fold, y_pred_fold)
#     roc_auc = roc_auc_score(y_test_fold, y_pred_prob_fold)

#     f1_scores.append(f1)
#     auc_roc_scores.append(roc_auc)

#     # 각 폴드의 성능 출력
#     print(f"Fold {i + 1}: F1 Score = {f1:.4f}, AUC-ROC Score = {roc_auc:.4f}")

# # 전체 폴드에 대한 평균 성능 출력
# print(f"\nAverage F1 Score: {sum(f1_scores) / len(f1_scores):.4f}")
# print(f"Average AUC-ROC Score: {sum(auc_roc_scores) / len(auc_roc_scores):.4f}")


In [ ]:
# import pandas as pd
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score, roc_auc_score
# from sklearn.model_selection import train_test_split, GridSearchCV, KFold

# # 하이퍼파라미터 그리드 설정 (max_iter 값을 높게 설정)
# param_grid = {
#     'penalty': ['l1', 'l2'],                # 규제 타입 선택
#     'C': [0.01, 0.1, 1.0, 10.0, 100.0],      # 규제 강도 값
#     'fit_intercept': [True, False],          # 상수 항 포함 여부
#     'tol': [1e-4, 1e-3, 1e-2],               # 수렴 조건 허용 오차
#     'class_weight': [None, 'balanced'],      # 클래스별 가중치 설정
#     'max_iter': [200, 500, 1000],            # 최대 반복 횟수 (증가된 값)
#     'warm_start': [True, False]              # 이어서 학습 여부
# }

# # 로지스틱 회귀 모델 정의
# model = LogisticRegression(solver='liblinear', random_state=42)

# # KFold 설정
# kf = KFold(n_splits=5)

# # GridSearchCV 설정
# grid_search = GridSearchCV(
#     estimator=model,
#     param_grid=param_grid,
#     scoring='roc_auc',      # AUC-ROC 점수를 사용하여 성능 평가
#     cv=kf,                  # 교차 검증의 fold 수 설정
#     n_jobs=-1,              # 모든 CPU 코어 사용
#     verbose=2               # 상세 정보 출력
# )

# # 그리드 탐색을 통한 최적의 하이퍼파라미터 찾기
# grid_search.fit(X_train, y_train)

# # 최적의 하이퍼파라미터 및 성능 출력
# print("Best Hyperparameters:", grid_search.best_params_)
# print("Best AUC-ROC Score:", grid_search.best_score_)

# # 각 폴드별 성능 평가 및 파라미터 출력
# splits = kf.split(X_train)
# for i, (train_index, test_index) in enumerate(splits):
#     # 각 폴드의 훈련/검증 데이터 분할 (iloc 사용)
#     X_train_fold, y_train_fold = X_train.iloc[train_index], y_train.iloc[train_index]
#     X_test_fold, y_test_fold = X_train.iloc[test_index], y_train.iloc[test_index]

#     # 각 폴드에서 사용한 하이퍼파라미터 가져오기
#     best_params = grid_search.cv_results_['params'][i]

#     # 최적의 모델로 각 폴드의 데이터를 학습
#     model_fold = LogisticRegression(**best_params, solver='liblinear', random_state=42)
#     model_fold.fit(X_train_fold, y_train_fold)

#     # 검증 데이터로 예측 수행
#     y_pred_fold = model_fold.predict(X_test_fold)
#     y_pred_prob_fold = model_fold.predict_proba(X_test_fold)[:, 1]

#     # 성능 평가 (F1 점수 및 AUC-ROC 점수)
#     f1 = f1_score(y_test_fold, y_pred_fold)
#     roc_auc = roc_auc_score(y_test_fold, y_pred_prob_fold)

#     # 각 폴드의 성능 출력
#     print(f"Fold {i + 1}: F1 Score = {f1:.4f}, AUC-ROC Score = {roc_auc:.4f}, Parameters: {best_params}")

# # 전체 폴드에 대한 평균 성능 출력
# average_auc_roc_score = sum(grid_search.cv_results_['mean_test_score']) / len(grid_search.cv_results_['mean_test_score'])
# print(f"\nAverage AUC-ROC Score (from GridSearchCV): {average_auc_roc_score:.4f}")


In [ ]:
X_test


In [ ]:
# 최종 모델 학습 - 모든 훈련 데이터를 사용
# model = LogisticRegression(class_weight='balanced', max_iter=10000)
# model.fit(X_train, y_train)

# 테스트 데이터에 대한 예측 확률 값 추출
test_prob = model.predict_proba(X_test)[:, 1]

# 예측 확률 값 출력
print("Predicted probabilities for X_test:")
print(test_prob)

In [ ]:
# 제출 파일 저장
submission_df = pd.read_csv('sample_submission.csv')
submission_df.health_status = test_prob
submission_df.to_csv('submission.csv', index=False)